# Feature Enhancement (GCP)
Adds holiday and COVID flags, plus Holt-Winters forecast feature.
Outputs enhanced features to HDFS under /user/tiennd.

In [ ]:
import pandas as pd
import numpy as np
import holidays
from datetime import date
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import DoubleType, StructType, StructField

BASE_HDFS = "/user/tiennd3886"
FEATURE_PATH = f"{BASE_HDFS}/feature_engineering/demand_prediction_features_30m"
OUT_FEATURES = f"{BASE_HDFS}/feature_engineering/demand_prediction_features_30m_enhanced"
TARGET_COL = "pickup_demand_t1"

spark = (
    SparkSession.builder
    .appName("FeatureEnhancement_GCP")
    .master("yarn")
    .config("spark.submit.deployMode", "client")
    .config("spark.eventLog.enabled", "true")
    .config("spark.executor.instances", "3")
    .config("spark.executor.cores", "3")
    .config("spark.executor.memory", "6g")
    .config("spark.executor.memoryOverhead", "1g")
    .config("spark.driver.memory", "4g")
    .config("spark.driver.memoryOverhead", "1g")
    .config("spark.sql.shuffle.partitions", "96")
    .getOrCreate()
 )
spark.sparkContext.setLogLevel("WARN")

df_all = spark.read.parquet(FEATURE_PATH).cache()
print("Rows:", df_all.count())

In [ ]:
# Holiday + COVID flags
us_holidays = holidays.UnitedStates(years=range(2019, 2027))
nyc_special = {
    date(2020, 1, 1), date(2021, 1, 1), date(2022, 1, 1),
    date(2023, 1, 1), date(2024, 1, 1), date(2025, 1, 1),
    date(2019, 12, 31), date(2020, 12, 31), date(2021, 12, 31),
    date(2022, 12, 31), date(2023, 12, 31), date(2024, 12, 31),
    date(2019, 11, 3), date(2021, 11, 7), date(2022, 11, 6),
    date(2023, 11, 5), date(2024, 11, 3),
}
holiday_dates = set(us_holidays.keys()) | nyc_special
covid_lockdown_ranges = [(date(2020, 3, 22), date(2020, 6, 7))]
covid_partial_ranges = [
    (date(2020, 1, 1), date(2020, 3, 21)),
    (date(2020, 6, 8), date(2021, 5, 19)),
 ]

date_rows = []
for d in pd.date_range("2019-01-01", "2026-12-31"):
    dt = d.date()
    is_hol = int(dt in holiday_dates)
    is_covid_lock = int(any(s <= dt <= e for s, e in covid_lockdown_ranges))
    is_covid_part = int(any(s <= dt <= e for s, e in covid_partial_ranges))
    date_rows.append((d.strftime("%Y-%m-%d"), is_hol, is_covid_lock, is_covid_part))

date_schema = "date_str STRING, is_holiday INT, is_covid_lockdown INT, is_covid_partial INT"
date_lookup_df = spark.createDataFrame(date_rows, schema=date_schema)

df_flagged = (
    df_all
    .withColumn("date_str", F.date_format(F.col("pickup_bin_30m"), "yyyy-MM-dd"))
    .join(F.broadcast(date_lookup_df), on="date_str", how="left")
    .fillna({"is_holiday": 0, "is_covid_lockdown": 0, "is_covid_partial": 0})
    .withColumn("is_holiday", F.col("is_holiday").cast(DoubleType()))
    .withColumn("is_covid_lockdown", F.col("is_covid_lockdown").cast(DoubleType()))
    .withColumn("is_covid_partial", F.col("is_covid_partial").cast(DoubleType()))
    .drop("date_str")
)

In [ ]:
# Holt-Winters forecast feature per zone (seasonal=48 for 30m bins)
HW_SEASONAL_PERIODS = 48
HW_MAX_TRAIN = HW_SEASONAL_PERIODS * 52
HW_MIN_ROWS = HW_SEASONAL_PERIODS * 3
HW_MIN_TRAIN = HW_SEASONAL_PERIODS * 2

def add_hw_forecast(pdf):
    pdf = pdf.sort_values("pickup_bin_30m").reset_index(drop=True)
    y = pdf[TARGET_COL].astype(float).to_numpy()
    if len(y) < HW_MIN_ROWS:
        pdf["hw_forecast"] = y; return pdf
    train_mask = pdf["split"] == "train"
    y_train_full = y[train_mask.to_numpy()]
    if len(y_train_full) < HW_MIN_TRAIN:
        pdf["hw_forecast"] = y; return pdf
    try:
        from statsmodels.tsa.holtwinters import ExponentialSmoothing
        y_fit = y_train_full[-HW_MAX_TRAIN:]
        n_older = len(y_train_full) - len(y_fit)
        n_test = int((~train_mask).sum())
        hw = ExponentialSmoothing(y_fit, trend="add", seasonal="add", seasonal_periods=HW_SEASONAL_PERIODS, initialization_method="estimated")
        fitted = hw.fit(optimized=True)
        in_sample = np.asarray(fitted.fittedvalues)
        train_fc = np.concatenate([np.full(n_older, in_sample[0]), in_sample]) if n_older > 0 else in_sample
        test_fc = np.asarray(fitted.forecast(n_test)) if n_test > 0 else np.array([])
        forecast = np.concatenate([train_fc, test_fc]) if n_test > 0 else train_fc
        pdf["hw_forecast"] = np.maximum(forecast, 0.0)
    except Exception:
        pdf["hw_forecast"] = y
    return pdf

hw_schema = StructType(df_flagged.schema.fields + [StructField("hw_forecast", DoubleType(), True)])
df_hw = df_flagged.groupBy("PULocationID").applyInPandas(add_hw_forecast, schema=hw_schema).cache()
print("Enhanced rows:", df_hw.count())

In [ ]:
df_hw.write.mode("overwrite").parquet(OUT_FEATURES)
print("Saved:", OUT_FEATURES)
spark.catalog.clearCache()
spark.stop()